In [35]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
import operator
import math
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal

load_dotenv()

True

In [36]:

endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

In [37]:
class QuadState(TypedDict):
    a: int
    b:int
    c:int
    equation:str
    discriminant:float
    result:str



In [38]:
def show_equation(state:QuadState)->QuadState:
    equation=f" {state['a']}x^2 + {state['b']}x + {state['c']} "
    return {"equation":equation}

In [39]:
def calculate_discriminant(state:QuadState)->QuadState:
    discriminant=state["b"]**2-4*state["a"]*state["c"]
    return {"discriminant":discriminant}

In [40]:
def real_roots(state:QuadState)->QuadState:
    root1=(-state["b"]+math.sqrt(state["discriminant"]))/(2*state["a"])
    root2=(-state["b"]-math.sqrt(state["discriminant"]))/(2*state["a"])
    result= f'the roots are {root1} and {root2}'
    return {"result":result}

In [41]:
def repeated_roots(state:QuadState)->QuadState:
    root=(-state["b"])/(2*state["a"])
    result=f'the root is {root}'
    return {"result":result}

In [42]:
def no_real_roots(state:QuadState)->QuadState:
    result="the equation has no real roots"
    return {"result":result}
    

In [43]:
def check_condition(state:QuadState)-> Literal["real_roots","repeated_roots","no_real_roots"]:
    if state["discriminant"]>0:
        return "real_roots"
    elif state["discriminant"]==0:
        return "repeated_roots"
    else:
        return "no_real_roots"
    

In [44]:
graph=StateGraph(QuadState)

graph.add_node("show_equation",show_equation)
graph.add_node("calculate_discriminant",calculate_discriminant)
graph.add_node("real_roots",real_roots)
graph.add_node("repeated_roots",repeated_roots)
graph.add_node("no_real_roots",no_real_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")
graph.add_conditional_edges(
    "calculate_discriminant",
    check_condition
)
graph.add_edge("real_roots", END)
graph.add_edge("repeated_roots", END)
graph.add_edge("no_real_roots", END)

workflow=graph.compile()


In [45]:
initial_state={"a":4,"b":-5,"c":-4}
result=workflow.invoke(initial_state)
print(result)

{'a': 4, 'b': -5, 'c': -4, 'equation': ' 4x^2 + -5x + -4 ', 'discriminant': 89, 'result': 'the roots are 1.8042476415070754 and -0.5542476415070754'}
